<a href="https://colab.research.google.com/github/mhmdabwjazyt8-ship-it/09231170_ai_prog/blob/master/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS453 - Two-Week Individual Project Workbook
**Name:** Mohamed Muftah Abu jazia  
**Student ID:** 09231170  
**Project:** Campus Delivery Agent  
**Variant:** generated below in the supplied helper cell (road removed + target building).

Story mapping (names only, the graph is the supplied one): S = Main Gate (start), A = Library, B = Cafeteria, C = Lab, D = Admin Building, E = Dorms, F = Sports Hall, G = Engineering.

## 1. Task, PEAS and environment

**Task.** A delivery agent starts at the Main Gate (S) holding one parcel. It must drive along campus roads to the assigned destination building and deliver the parcel there.

**Input:** the student ID (which fixes the closed road and the target), the road graph, the start location.  
**Output:** a route (list of locations) with the minimum number of moves, and the simulator state after executing the route and the delivery action.  
**Success criteria:** the route only uses open roads, ends at the target, the parcel status becomes *delivered*, and the move count is minimal (BFS and A* agree).

**PEAS**

| | |
|---|---|
| **Performance measure** | Parcel delivered at the assigned target; fewest moves (every road costs 1); no illegal actions attempted. |
| **Environment** | Campus road network of 8 locations (S, A-G) with undirected unit-cost roads, one road closed by the variant, one parcel, one target building. |
| **Actuators** | Move to an adjacent location along an open road; deliver the parcel. |
| **Sensors** | Current location, the open roads leaving it, parcel status (delivered or not), the target. |

**Environment classification**

- **Fully observable:** the agent knows the whole map, its own location, the parcel status and the target.
- **Deterministic:** a move always reaches the chosen neighbour; delivery always succeeds when legal.
- **Sequential:** earlier moves decide which later moves/deliveries are possible (delivery is only legal at the target), so it is not a one-shot episode.
- **Static:** the map and the closed road do not change while the agent plans or acts.
- **Discrete:** finite set of locations, discrete actions and discrete time steps.
- **Single agent:** one delivery agent, no other agents.

**Three scope limits**

1. Pure Python simulation: no real vehicle, GPS, sensors or UI.
2. All roads cost 1: no distances, traffic, delays or weighted roads (so BFS is optimal and the supplied heuristic is valid).
3. One parcel, one fixed target and one fixed road closure; no new closures or several deliveries while running.

## 2. Representation

- **State:** `(location, delivered)`, e.g. `("S", False)`. It contains task status as well as location.
- **Initial state:** `("S", False)`: at the Main Gate with the parcel not yet delivered.
- **Legal actions in state `(loc, delivered)`:**
  - `("move", n)` for every neighbour `n` of `loc` in the (variant) graph;
  - `("deliver", None)` only if `loc == target` and `delivered == False`.
- **Result / transition:**
  - `("move", n)` gives `(n, delivered)`;
  - `("deliver", None)` gives `(target, True)`.
- **Goal test:** for route search, the goal is reaching the target location; for the whole task, the goal state is `(target, True)` (delivered).
- **Costs:** each move costs 1. The delivery action costs 0 because it is a service, not travel, so the path cost is the number of moves.
- **Service legality and repeated service:** delivery is rejected (clear `ValueError`) away from the target, and also if the parcel was already delivered.
- **Initially complete?** No: the parcel starts undelivered and the start S is not the target.

In [11]:
# Supplied data/helper: cite CS453 two-week project workbook if adapted.
from copy import deepcopy
import platform

REFERENCE_GRAPH = {
    "S": ["A", "B"],
    "A": ["S", "C", "D"],
    "B": ["S", "E"],
    "C": ["A", "F"],
    "D": ["A", "G"],
    "E": ["B", "G"],
    "F": ["C", "G"],
    "G": ["D", "E", "F"],
}
VARIANTS = [("A", "D", "G"), ("B", "E", "F"),
            ("C", "F", "E"), ("E", "G", "D")]

def make_variant(student_id):
    if not str(student_id).strip():
        raise ValueError("Enter your student ID first")
    variant = sum(ord(ch) for ch in str(student_id).strip()) % 4
    graph = deepcopy(REFERENCE_GRAPH)
    u, v, goal = VARIANTS[variant]
    graph[u].remove(v)
    graph[v].remove(u)
    return {"variant": variant, "graph": graph, "start": "S",
            "goal": goal, "removed_edge": (u, v)}

STUDENT_ID = "09231170"  # student ID as text, preserving leading zeros.
if STUDENT_ID.strip():
    assigned = make_variant(STUDENT_ID)
    print(assigned)
else:
    print("Unfinished template: enter your ID to generate your variant.")
print("Python:", platform.python_version())

{'variant': 3, 'graph': {'S': ['A', 'B'], 'A': ['S', 'C', 'D'], 'B': ['S', 'E'], 'C': ['A', 'F'], 'D': ['A', 'G'], 'E': ['B'], 'F': ['C', 'G'], 'G': ['D', 'F']}, 'start': 'S', 'goal': 'D', 'removed_edge': ('E', 'G')}
Python: 3.13.18


In [12]:
# Convenient names for the assigned variant (the graph is never modified below).
GRAPH = assigned["graph"]
START = assigned["start"]
TARGET = assigned["goal"]
print(f"Variant v = {assigned['variant']}")
print(f"Closed road: {assigned['removed_edge'][0]}-{assigned['removed_edge'][1]}")
print(f"Start: {START}   Target building: {TARGET}")
for node, nbrs in GRAPH.items():
    print(f"  {node}: {nbrs}")

Variant v = 3
Closed road: E-G
Start: S   Target building: D
  S: ['A', 'B']
  A: ['S', 'C', 'D']
  B: ['S', 'E']
  C: ['A', 'F']
  D: ['A', 'G']
  E: ['B']
  F: ['C', 'G']
  G: ['D', 'F']


In [13]:
# Supplied heuristic data (CS453 workbook). HEURISTICS[target][node] = shortest distance
# in the ORIGINAL graph (before the road closure), so it is a lower bound on the variant graph.
HEURISTICS = {'A': {'A': 0, 'B': 2, 'C': 1, 'D': 1, 'E': 3, 'F': 2, 'G': 2, 'S': 1}, 'B': {'A': 2, 'B': 0, 'C': 3, 'D': 3, 'E': 1, 'F': 3, 'G': 2, 'S': 1}, 'C': {'A': 1, 'B': 3, 'C': 0, 'D': 2, 'E': 3, 'F': 1, 'G': 2, 'S': 2}, 'D': {'A': 1, 'B': 3, 'C': 2, 'D': 0, 'E': 2, 'F': 2, 'G': 1, 'S': 2}, 'E': {'A': 3, 'B': 1, 'C': 3, 'D': 2, 'E': 0, 'F': 2, 'G': 1, 'S': 2}, 'F': {'A': 2, 'B': 3, 'C': 1, 'D': 2, 'E': 2, 'F': 0, 'G': 1, 'S': 3}, 'G': {'A': 2, 'B': 2, 'C': 2, 'D': 1, 'E': 1, 'F': 1, 'G': 0, 'S': 3}, 'S': {'A': 1, 'B': 1, 'C': 2, 'D': 2, 'E': 2, 'F': 3, 'G': 3, 'S': 0}}
print("Heuristic tables available for targets:", ", ".join(sorted(HEURISTICS)))
print("h for my target", TARGET, "->", HEURISTICS[TARGET])

Heuristic tables available for targets: A, B, C, D, E, F, G, S
h for my target D -> {'A': 1, 'B': 3, 'C': 2, 'D': 0, 'E': 2, 'F': 2, 'G': 1, 'S': 2}


## 4. Implement and test the simulator

The simulator knows the rules (legal actions, state updates, service). It does **not** search. States are immutable tuples `(location, delivered)`, so no shared data is mutated between tests.

In [22]:
def initial_state(start=START):
    """Agent at `start`, parcel not yet delivered."""
    if start not in GRAPH:
        raise ValueError(f"Invalid start location: {start!r}")
    return (start, False)

def legal_actions(state, graph=GRAPH, target=TARGET):
    """All legal actions in `state`: moves along open roads, and delivery if allowed."""
    location, delivered = state
    if location not in graph:
        raise ValueError(f"Invalid location: {location!r}")
    actions = [("move", nbr) for nbr in graph[location]]
    if location == target and not delivered:
        actions.append(("deliver", None))
    return actions

def apply_action(state, action, graph=GRAPH, target=TARGET):
    """Return (new_state, cost). Raise ValueError with a clear message on an illegal action."""
    location, delivered = state
    if location not in graph:
        raise ValueError(f"Invalid location: {location!r}")
    kind, arg = action
    if kind == "move":
        if arg not in graph:
            raise ValueError(f"Invalid move: {arg!r} is not a campus location")
        if arg not in graph[location]:
            raise ValueError(f"Illegal move: no open road {location}-{arg}")
        return (arg, delivered), 1                    # a move costs 1
    if kind == "deliver":
        if location != target:
            raise ValueError(f"Illegal delivery at {location}: parcel must be delivered at {target}")
        if delivered:
            raise ValueError("Illegal delivery: parcel was already delivered")
        return (location, True), 0                    # service action: not travel, costs 0
    raise ValueError(f"Unknown action: {action!r}")

def is_task_complete(state, target=TARGET):
    location, delivered = state
    return delivered and location == target

def execute_plan(route, graph=GRAPH, target=TARGET, verbose=True):
    """Drive along `route` (list of locations) then deliver. Returns (final_state, total_cost)."""
    state = initial_state(route[0])
    total = 0
    if verbose:
        print("start  ", state)
    for nxt in route[1:]:
        state, cost = apply_action(state, ("move", nxt), graph, target)
        total += cost
        if verbose:
            print(f"move {nxt} ->", state)
    state, cost = apply_action(state, ("deliver", None), graph, target)
    total += cost
    if verbose:
        print("deliver ->", state, "| task complete:", is_task_complete(state, target), "| moves:", total)
    return state, total

In [23]:
# Quick simulator checks (illegal actions must be rejected clearly).
s0 = initial_state()
print("initial state:", s0, "| complete?", is_task_complete(s0))
print("legal at start:", legal_actions(s0))

for bad in [("move", "G"),        # no road S-G
            ("move", "Z"),        # not a location
            ("deliver", None)]:   # delivery away from the target
    try:
        apply_action(s0, bad)
    except ValueError as e:
        print("rejected:", bad, "->", e)

# The closed road really is closed: E-G must not be usable in my variant.
u, v = assigned["removed_edge"]
try:
    apply_action((u, False), ("move", v))
except ValueError as e:
    print("closed road rejected ->", e)

# Repeated delivery is rejected.
done_state, _ = apply_action((TARGET, False), ("deliver", None))
try:
    apply_action(done_state, ("deliver", None))
except ValueError as e:
    print("repeat delivery rejected ->", e)

# The shared map was not mutated.
print("graph unchanged:", GRAPH == assigned["graph"])

initial state: ('S', False) | complete? False
legal at start: [('move', 'A'), ('move', 'B')]
rejected: ('move', 'G') -> Illegal move: no open road S-G
rejected: ('move', 'Z') -> Invalid move: 'Z' is not a campus location
rejected: ('deliver', None) -> Illegal delivery at S: parcel must be delivered at D
closed road rejected -> Illegal move: no open road E-G
repeat delivery rejected -> Illegal delivery: parcel was already delivered
graph unchanged: True


## 5. Uninformed search: BFS

**Why BFS is uninformed.** It orders nodes only by the order they were discovered (a FIFO queue), using no information about where the goal is; it would behave the same for any goal.

**Why it returns a minimum-move path here.** Every road costs 1 and BFS expands nodes level by level (all nodes at distance k before any at distance k+1). The first time a node is reached is therefore along a path with the fewest moves; later discoveries of the same node can never be shorter, so repeated states can be ignored safely.

**Implementation.** FIFO queue (`deque`), a `parent` dict that doubles as the visited set (repeated-state tracking), and route reconstruction by following parents back from the goal. Edge cases: start = goal returns `[start]`, unreachable goal returns `None`, unknown locations raise `ValueError`.

In [16]:
from collections import deque

def reconstruct_path(parent, goal):
    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]

def bfs(graph, start, goal):
    """Return (route, expansion_order). route is None if the goal is unreachable."""
    for name, node in (("start", start), ("goal", goal)):
        if node not in graph:
            raise ValueError(f"Invalid {name} location: {node!r}")
    if start == goal:
        return [start], []
    frontier = deque([start])          # FIFO queue
    parent = {start: None}             # visited set + parent links
    order = []
    while frontier:
        current = frontier.popleft()
        order.append(current)
        if current == goal:
            return reconstruct_path(parent, goal), order
        for nbr in graph[current]:
            if nbr not in parent:      # repeated-state check
                parent[nbr] = current
                frontier.append(nbr)
    return None, order

In [17]:
# Assigned mission with BFS
bfs_route, bfs_order = bfs(GRAPH, START, TARGET)
print("BFS expansion order:", bfs_order)
print("BFS route:", " -> ".join(bfs_route), "| moves:", len(bfs_route) - 1)
print()
bfs_state, bfs_cost = execute_plan(bfs_route)

BFS expansion order: ['S', 'A', 'B', 'C', 'D']
BFS route: S -> A -> D | moves: 2

start   ('S', False)
move A -> ('A', False)
move D -> ('D', False)
deliver -> ('D', True) | task complete: True | moves: 2


## 6. Informed search: A*

**g and h.** `g(n)` is the cost already paid from the start to `n` (number of moves so far). `h(n)` is the estimated remaining cost from `n` to the goal, taken from the supplied `HEURISTICS[target]` table. A* always expands the frontier node with the smallest `f(n) = g(n) + h(n)`.

**Admissibility on my variant.** The table stores shortest distances in the *original* graph. My variant only removes one road (E-G). Removing a road can only keep distances the same or make them longer, so the true remaining cost in my graph is always >= `h`. Therefore `h` never overestimates (it is admissible) and `h(goal) = 0`. For example `h(E) = 2` while the real route from E now is E-B-S-A-D = 4 moves.

**When it would need revising.** If I added shortcuts or cheaper/weighted roads, true distances could fall below the table values, so `h` could overestimate. Then I would have to recompute/justify `h`, or use `h = 0` (which makes A* behave like uniform-cost search).

**Implementation details.** `heapq` priority queue ordered by `f`, a counter as tie-breaker, `best_g` to keep the cheapest known cost per node (a cheaper path found later replaces the old entry; outdated queue entries are skipped), `parent` links for reconstruction, and the **goal test when the node is removed from the frontier** (not when it is generated), which is what guarantees optimality.

In [18]:
import heapq
from itertools import count

def astar(graph, start, goal, h):
    """Return (route, cost, expansion_order). route is None if the goal is unreachable."""
    for name, node in (("start", start), ("goal", goal)):
        if node not in graph:
            raise ValueError(f"Invalid {name} location: {node!r}")
    if start == goal:
        return [start], 0, []
    tie = count()                                    # tie-breaker so heap never compares nodes
    frontier = [(h[start], 0, next(tie), start)]     # entries: (f, g, tie, node)
    best_g = {start: 0}
    parent = {start: None}
    expanded = set()
    order = []
    while frontier:
        f, g, _, current = heapq.heappop(frontier)
        if current in expanded or g > best_g[current]:
            continue                                  # stale entry
        expanded.add(current)
        order.append(current)
        if current == goal:                           # goal test on removal from the frontier
            return reconstruct_path(parent, goal), g, order
        for nbr in graph[current]:
            new_g = g + 1                             # unit-cost roads
            if new_g < best_g.get(nbr, float("inf")): # best-cost update
                best_g[nbr] = new_g
                parent[nbr] = current
                heapq.heappush(frontier, (new_g + h[nbr], new_g, next(tie), nbr))
    return None, None, order

In [19]:
# Assigned mission with A*
h = HEURISTICS[TARGET]
astar_route, astar_cost, astar_order = astar(GRAPH, START, TARGET, h)
print("A* expansion order:", astar_order)
print("A* route:", " -> ".join(astar_route), "| g(goal):", astar_cost)
print("f at start = g + h =", 0 + h[START])
print()
astar_state, astar_total = execute_plan(astar_route)

print()
print("BFS moves:", len(bfs_route) - 1, "| A* moves:", astar_cost, "| same minimum:", len(bfs_route) - 1 == astar_cost)
print("Same route?", bfs_route == astar_route, "(routes may differ; the move count must not)")

A* expansion order: ['S', 'A', 'D']
A* route: S -> A -> D | g(goal): 2
f at start = g + h = 2

start   ('S', False)
move A -> ('A', False)
move D -> ('D', False)
deliver -> ('D', True) | task complete: True | moves: 2

BFS moves: 2 | A* moves: 2 | same minimum: True
Same route? True (routes may differ; the move count must not)


In [20]:
# Edge cases required by the task: start = goal, unreachable goal, invalid locations.
print("start = goal  BFS:", bfs(GRAPH, "S", "S"))
print("start = goal  A* :", astar(GRAPH, "S", "S", HEURISTICS["S"]))

# Unreachable: a small copy of the graph with S cut off from everything (original GRAPH untouched).
cut = deepcopy(GRAPH)
for nbr in cut["S"]:
    cut[nbr].remove("S")
cut["S"] = []
print("unreachable   BFS:", bfs(cut, "S", TARGET)[0])
print("unreachable   A* :", astar(cut, "S", TARGET, HEURISTICS[TARGET])[0])

for fn_name, call in [("BFS invalid start", lambda: bfs(GRAPH, "Z", TARGET)),
                      ("BFS invalid goal", lambda: bfs(GRAPH, START, "Z")),
                      ("A* invalid start", lambda: astar(GRAPH, "Z", TARGET, HEURISTICS[TARGET])),
                      ("A* invalid goal", lambda: astar(GRAPH, START, "Z", HEURISTICS[TARGET]))]:
    try:
        call()
    except ValueError as e:
        print(fn_name, "->", e)

# A* with a different target also matches BFS (sanity check that the code is general).
for t in "ABCDEFG":
    r1, _ = bfs(GRAPH, "S", t)
    r2, c2, _ = astar(GRAPH, "S", t, HEURISTICS[t])
    assert len(r1) - 1 == c2, (t, r1, r2)
print("BFS and A* agree on move count for every target in my variant graph.")

start = goal  BFS: (['S'], [])
start = goal  A* : (['S'], 0, [])
unreachable   BFS: None
unreachable   A* : None
BFS invalid start -> Invalid start location: 'Z'
BFS invalid goal -> Invalid goal location: 'Z'
A* invalid start -> Invalid start location: 'Z'
A* invalid goal -> Invalid goal location: 'Z'
BFS and A* agree on move count for every target in my variant graph.


## 10. Sources, assistance and questions

**Sources / adapted code.** The graph, variant helper, and heuristic tables are supplied by the CS453 two-week project workbook. BFS and A* follow the ideas of the supplied course notebooks (graph-search and informed-search STUDENT notebooks) and the standard textbook algorithms (Russell & Norvig, *Artificial Intelligence: A Modern Approach*); the code in this notebook was written for the Campus Delivery story (state = location + delivered, separate simulator).

**AI assistance disclosure.** Tool used: Claude (Anthropic). Help received: explanation of the task and variant, debugging minor code errors, and formatting the markdown text. My verification: I ran the notebook from a fresh Colab session, checked the variant (v = 3, closed road E-G, target D) by hand, checked that BFS and A* both give the route S -> A -> D (2 moves), and tested the illegal-action and edge cases.


## 11. Submission checklist

- [x] Identity, chosen project and generated variant recorded
- [x] Task 1: PEAS and environment explained
- [x] Task 2: state, legal actions, transition, goal and costs defined
- [x] Task 4: legal movement, service and task goal implemented
- [x] Task 5: working BFS and assigned mission output
- [x] Task 6: working A*, admissible heuristic explanation and assigned mission output
- [x] Sources and assistance disclosed
- [ ] README with task/variant/files/dependencies/run steps
- [ ] Clean restart/run-all checked and saved
- [ ] project.ipynb and README.md saved in my private GitHub repository
- [ ] Professor invited; repository URL and final commit URL submitted in Classroom